In [1]:
## 0 – Imports & basic configuration
import os, json, math, re
from pathlib import Path
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from nltk.stem import WordNetLemmatizer
from discovery_utils.utils.llm import batch_check

try:
    from discovery_heat_pump_futures import PROJECT_DIR
except ModuleNotFoundError:
    PROJECT_DIR = Path(".").resolve()     


/home/pascualdiego/projects/DiscoveryHP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
client = OpenAI()

In [3]:
# # ##Testing Step 1: Check OpenAI API directly##
# def test_openai_api():
#     """Test if OpenAI API is working"""
#     try:
#         from openai import OpenAI
#         client = OpenAI()
        
#         # Test with a simple completion
#         response = client.chat.completions.create(
#             model="gpt-4o-mini",
#             messages=[{"role": "user", "content": "Say 'API is working'"}],
#             max_tokens=10
#         )
#         print("✓ OpenAI API is working:", response.choices[0].message.content)
#         return True
#     except Exception as e:
#         print("✗ OpenAI API error:", e)
#         return False

# # Test API
# api_works = test_openai_api()

In [4]:
# # ##Testing Step 2: Check batch_check module##
# def inspect_batch_check():
#     """Inspect the batch_check module to understand how it works"""
#     from discovery_utils.utils.llm import batch_check
    
#     print("\nBatch check module info:")
#     print(f"Module location: {batch_check.__file__}")
    
#     # Check if LLMProcessor has a different way to wait for results
#     processor_methods = [m for m in dir(batch_check.LLMProcessor) if not m.startswith('_')]
#     print(f"LLMProcessor methods: {processor_methods}")
    
#     return batch_check

# # Inspect module
# batch_module = inspect_batch_check()

In [5]:
# # ##Testing Step 3: Try a simpler approach with manual processing##
# def process_with_direct_api(documents, system_message, fields):
#     """Process documents directly with OpenAI API instead of batch_check"""
#     from openai import OpenAI
#     client = OpenAI()
    
#     results = []
    
#     for doc_id, text in documents.items():
#         print(f"Processing {doc_id}...")
        
#         # Create the field descriptions for the prompt
#         field_desc = "\n".join([f"- {f['name']}: {f['description']}" for f in fields])
        
#         prompt = f"""
# {system_message}

# Document to analyze:
# {text[:1000]}  # First 1000 chars to avoid token limits

# Please provide a JSON response with these fields:
# {field_desc}
# """
        
#         try:
#             response = client.chat.completions.create(
#                 model="gpt-4o-mini",
#                 messages=[
#                     {"role": "system", "content": system_message},
#                     {"role": "user", "content": prompt}
#                 ],
#                 response_format={"type": "json_object"},
#                 temperature=0
#             )
            
#             result = json.loads(response.choices[0].message.content)
#             result['_id'] = doc_id
#             results.append(result)
            
#         except Exception as e:
#             print(f"Error processing {doc_id}: {e}")
            
#     return results

In [6]:
# # ##Testing Step 4: Alternative - Check if batch_check needs initialization##
# def check_batch_check_logs():
#     """Look for log files or output from batch_check"""
#     import os
    
#     # Check for log files
#     log_files = list(Path(".").glob("*.log"))
#     if log_files:
#         print("Found log files:")
#         for log in log_files:
#             print(f"- {log}")
            
#     # Check environment variables that might be needed
#     env_vars = ["OPENAI_API_KEY", "LANGFUSE_PUBLIC_KEY", "BATCH_CHECK_OUTPUT_DIR"]
#     print("\nEnvironment variables:")
#     for var in env_vars:
#         value = os.getenv(var)
#         if value:
#             print(f"✓ {var} is set")
#         else:
#             print(f"✗ {var} is not set")

# # Run checks
# check_batch_check_logs()

In [7]:
# # ## Testing Step 5: Try the existing classified files instead##
# OUTPUT_DIR = PROJECT_DIR / "outputs"
# def use_existing_results():
#     """Load and work with your existing classified files"""
    
#     # You have many classified files already - let's use them!
#     print("\nYou already have classified results!")
    
#     # Load the most complete files
#     patents_df = pd.read_json(OUTPUT_DIR / "fullset_classified_patents_CEscore_v2.jsonl", lines=True)
#     papers_df = pd.read_json(OUTPUT_DIR / "papers3_classified.jsonl", lines=True)
    
#     print(f"\nLoaded existing results:")
#     print(f"- Patents: {len(patents_df)} documents")
#     print(f"- Papers: {len(papers_df)} documents")
    
#     # Check what columns they have
#     print(f"\nPatent columns: {list(patents_df.columns)}")
#     print(f"\nPaper columns: {list(papers_df.columns)}")
    
#     # If they don't have all the new fields, we can add them
#     return patents_df, papers_df

# # Load existing results
# patents_df, papers_df = use_existing_results()

In [8]:
## 1 – Load the full patent & paper datasets 
OPENALEX_URL = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_openalex.csv"
PATENT_URL   = "https://discovery-hub-open-data.s3.eu-west-2.amazonaws.com/future_heat_pumps/heat_pumps_patents.json"

openalex_df = pd.read_csv(OPENALEX_URL, low_memory=False)
patents_df  = pd.read_json(PATENT_URL, lines=True)

def format_title_abstract(df, title="title", abstract="abstract"):
    df = df.copy()
    df["title_abstract"] = (
        "TITLE: " + df[title].fillna("").str.lower() +
        " ABSTRACT: " + df[abstract].fillna("").str.lower()
    )
    return df

openalex_df = format_title_abstract(openalex_df)
patents_df  = format_title_abstract(patents_df)

print(f"Loaded {len(openalex_df):,} papers | {len(patents_df):,} patents")


Loaded 33,599 papers | 45,957 patents


In [9]:
## 2.1 – Heat-pump taxonomy & keyword universe
CATEGORIES = {
    "1.1": "Compressors", 
    "1.2": "Refrigerants", 
    "1.3": "Heat-exchangers", 
    "1.4": "Motor & Drives", 
    "1.5": "Lubrication & oil management",
    "2.1": "Elastocaloric", 
    "2.2": "Electrocaloric", 
    "2.3": "Magnetocaloric", 
    "2.4": "Ionocaloric", 
    "2.5": "Barocaloric",
    "2.6": "Thermoelectric", 
    "2.7": "Electro- & Chemisorption", 
    "2.8": "Thermoacoustic", 
    "2.9": "Sorption / Absorption", 
    "2.10": "Hybrid & cascade",
    "3.1": "Commissioning & installation",  # Changed from "Topology & configuration"
    "3.2": "Controls & optimisation", 
    "3.3": "Operational integration",
    "4.1": "Flexible cycles", 
    "4.2": "Defrost & icing mitigation", 
    "4.3": "Thermal storage",
    "5.1": "Design-for-disassembly", 
    "5.2": "Modular assemblies", 
    "5.3": "Recycled materials", 
    "5.4": "Additive manufacturing", 
    "5.5": "Predictive maintenance"
}

# ---- 2.2 KEYWORD UNIVERSE ---------------------------------------------------
KWS = {
    "1.1": ["scroll", "rotary", "vane", "twin-screw", "reciprocating", "isothermal compressor", "oil-free", "variable-speed", "economiser", "compressor-modulation", "two-stage"],
    "1.2": ["R32", "R454B", "R290", "propane", "CO₂", "carbon dioxide", "low-GWP", "natural refrigerant", "azeotrope", "zeotropic", "ionic liquid"],
    "1.3": ["micro-channel", "plate heat exchanger", "fin-tube", "enthalpy exchanger", "phase-change heat exchanger", "anti-fouling", "frost-free", "3-D printed"],
    "1.4": ["IPM motor", "SiC inverter", "PMSM", "sensor-less", "flux-weakening"],
    "1.5": ["oil-separator", "mist injection", "CRII", "low-viscosity oil"],
    "2.1": ["elastocaloric", "shape-memory", "Ni-Ti"], 
    "2.2": ["electrocaloric", "ferroelectric"], 
    "2.3": ["magnetocaloric", "gadolinium"],
    "2.4": ["ionocaloric", "ion-solvation"], 
    "2.5": ["barocaloric"], 
    "2.6": ["thermoelectric", "Peltier", "Seebeck"],
    "2.7": ["electrochemical compressor", "chemisorption"], 
    "2.8": ["thermoacoustic"],
    "2.9": ["adsorption heat pump", "absorption heat pump", "lithium bromide"], 
    "2.10": ["cascade heat pump", "hybrid heat pump"],
    "3.1": ["commissioning", "installation", "setup", "configuration", "system design", "cascade", "booster", "trans-critical", "transcritical", "bi-valent", "bivalent", "ground-source", "ground source"],
    "3.2": ["model predictive control", "MPC", "PID", "fault detection", "digital twin", "optimization", "machine learning", "AI control", "smart control"],
    "3.3": ["smart-grid", "thermal district network", "hybrid boiler"], 
    "4.1": ["ejector cycle", "regenerative cycle", "parallel-compressor"],
    "4.2": ["defrost", "icing sensor", "hot-gas bypass", "nano-coating"], 
    "4.3": ["PCM storage", "phase change material", "heat battery", "stratified tank"],
    "5.1": ["tool-less", "snap-fit", "reversible adhesive", "design-for-disassembly", "DfD"], 
    "5.2": ["modular cartridge", "remanufacture", "refurbish", "life-extension"],
    "5.3": ["recycled", "bio-polymer", "PCR plastic", "reclaimed copper", "low-GWP foam"], 
    "5.4": ["additive manufacturing", "3-D print", "near-net shape", "binder-jet"],
    "5.5": ["predictive maintenance", "condition monitoring", "service-as-a-product", "serviceability", "repare", "restore"]
}

#Old KW2CAT = {kw.lower(): cat for cat, kwlist in KWS.items() for kw in kwlist}

#Rosie's feedback Slack
# Normalize keywords:
def normalize_keyword(kw):
    """Normalize keywords to handle variants (e.g., CO₂ -> co2)"""
    return kw.lower().replace("₂", "2").replace("-", " ").replace("_", " ")

# Updated KW2CAT creation to use normalized keywords:
KW2CAT = {}
for cat, kwlist in KWS.items():
    for kw in kwlist:
        KW2CAT[normalize_keyword(kw)] = cat

# Function to check if text contains any keyword:
def contains_keyword(text):
    """Check if text contains any heat pump keyword"""
    text_normalized = normalize_keyword(text)
    return any(normalize_keyword(kw) in text_normalized for kw_list in KWS.values() for kw in kw_list)


In [ ]:
# #Testing
# print(list(CATEGORIES.values()))

['Compressors', 'Refrigerants', 'Heat-exchangers', 'Motor & Drives', 'Lubrication & oil management', 'Elastocaloric', 'Electrocaloric', 'Magnetocaloric', 'Ionocaloric', 'Barocaloric', 'Thermoelectric', 'Electro- & Chemisorption', 'Thermoacoustic', 'Sorption / Absorption', 'Hybrid & cascade', 'Commissioning & installation', 'Controls & optimisation', 'Operational integration', 'Flexible cycles', 'Defrost & icing mitigation', 'Thermal storage', 'Design-for-disassembly', 'Modular assemblies', 'Recycled materials', 'Additive manufacturing', 'Predictive maintenance']


In [11]:
## 3 – ISO-59004 circularity heuristic (To be refined with the standard, working on it)
ISO_LEVERS = {
    "maintain": [
        "predictive maintenance", "preventive maintenance", "condition monitoring",
        "remote diagnostics", "fault detection", "service-as-a-product", "self-healing system"
    ],
    "reuse": [
        "remanufacture", "remanufacturing", "refurbish", "refurbished", "repair", "repairability",
        "cartridge", "replaceable module", "core exchange", "component reuse", "second life"
    ],
    "recycle": [
        "recycled", "recycling", "reclaim", "reclaimed", "material recovery",
        "mono-material", "material loop", "mechanical recycling", "chemical recycling", "closed-loop"
    ],
    "reduce": [
        "near-net shape", "additive manufacturing", "3d printing", "lightweight design",
        "material efficiency", "yield improvement", "low-waste", "minimal material use", "net-shape forming"
    ],
    "regenerate": [
        "bio-based", "biobased", "bio-polymer", "biopolymer", "biocomposite",
        "renewable feedstock", "natural material", "biodegradable", "biomaterial", "plant-derived"
    ]
}               

lemmatizer = WordNetLemmatizer()

def _norm(t):  return [lemmatizer.lemmatize(w) for w in re.findall(r"\b\w+\b", t.lower())]

def iso_score(text: str) -> int:
    toks = " ".join(_norm(text))
    hits = []
    for kws in ISO_LEVERS.values():
        m = sum(1 for kw in kws if kw in toks)
        hits.append(0 if m == 0 else 1 if m == 1 else 2 if m == 2 else 3)
    return math.ceil(sum(hits)/5)


In [12]:
# ## 4 – System message for GPT
#  SYSTEM_MESSAGE = f"""
#  You are a sustainable-heating technology analyst. …
#  4. Score 0-3 for:
#   # a) cost_reduction_potential
#   # b) efficiency_gain_potential
#   # c) circularity_score (ISO 59004 lever heuristic)
#  Return ONLY JSON with the six fields requested.
#  """


In [13]:
#2.3 Category Explanation for accuracy
CATEGORY_EXPLANATIONS = {
    # Traditional Components 
    "1.1": "Compressors: Devices that compress refrigerant, including scroll, rotary, reciprocating types",
    "1.2": "Refrigerants: Working fluids that undergo phase changes, including natural and synthetic options",
    "1.3": "Heat-exchangers: Components for heat transfer between refrigerant and air/water",
    "1.4": "Motor & Drives: Electric motors and control systems that power the compressor",
    "1.5": "Lubrication & oil management: Systems for compressor lubrication and oil circulation",
    
    # Non-traditional Technologies 
    "2.1": "Elastocaloric: Uses stress-induced phase transitions in shape-memory alloys",
    "2.2": "Electrocaloric: Uses electric field-induced temperature changes in ferroelectric materials",
    "2.3": "Magnetocaloric: Uses magnetic field-induced temperature changes (e.g., gadolinium)",
    "2.4": "Ionocaloric: Uses ion dissolution/crystallization for cooling",
    "2.5": "Barocaloric: Uses pressure-induced phase transitions",
    "2.6": "Thermoelectric: Uses Peltier/Seebeck effects for solid-state cooling",
    "2.7": "Electro- & Chemisorption: Uses electrochemical processes or chemical absorption",
    "2.8": "Thermoacoustic: Uses acoustic waves to create temperature differences",
    "2.9": "Sorption/Absorption: Uses chemical absorption (e.g., lithium bromide-water)",
    "2.10": "Hybrid & cascade: Combines multiple technologies or stages",
    
    # System Design 
    "3.1": "Commissioning & installation: System setup, configuration, and initial startup procedures",
    "3.2": "Controls & optimisation: Control systems, algorithms, and performance optimization",
    "3.3": "Operational integration: Integration with buildings, grids, or other systems",
    
    # System Enhancements 
    "4.1": "Flexible cycles: Advanced refrigeration cycles for improved performance",
    "4.2": "Defrost & icing mitigation: Technologies to prevent or remove ice formation",
    "4.3": "Thermal storage: Heat/cold storage systems for load shifting",
    
    # Circular Economy 
    "5.1": "Design-for-disassembly: Products designed for easy dismantling and component recovery",
    "5.2": "Modular assemblies: Replaceable/upgradeable modules for extended product life",
    "5.3": "Recycled materials: Use of recycled or sustainable materials in manufacturing",
    "5.4": "Additive manufacturing: 3D printing and other advanced manufacturing techniques",
    "5.5": "Predictive maintenance: IoT and AI-based maintenance to extend equipment life"
}

In [14]:
# Karlis's feedback: Defining APPLICATION_INDICATORS (as a dictionary)
APPLICATION_INDICATORS = {
    "domestic": ["residential", "home", "household", "domestic", "small-scale", "<20kW", "single-family", "apartment"],
    "industrial": ["commercial", "industrial", "large-scale", "district", "process heat", ">100kW", "manufacturing", "warehouse"]
}

# Function to format the indicators
def format_application_indicators(indicators_dict):
    """Format application indicators for display in prompt"""
    lines = []
    for app_type, keywords in indicators_dict.items():
        lines.append(f"- {app_type.capitalize()}: {', '.join(keywords)}")
    return "\n".join(lines)

In [15]:
#Max's feedback: Market readiness. 
TRL_DEFINITIONS = """
TRL 1: Basic principles observed and reported
TRL 2: Technology concept formulated and validated
TRL 3: Applied research and proof of concept
TRL 4: Component-level validation in lab environment
TRL 5: Prototype tested in intended environment
TRL 6: Prototype system tested, close to expected performance
TRL 7: Demonstration system at pre-commercial scale
TRL 8: First commercial system, manufacturing issues resolved
TRL 9: Full commercial application available to consumers
"""

In [16]:
# # 4_v2 Updated SYSTEM_MESSAGE for GPT
# SYSTEM_MESSAGE = f"""
# You are a sustainable-heating technology analyst evaluating heat pump innovations.

# Categories to classify into:
# {json.dumps(CATEGORIES, indent=2)}

# Technology Readiness Levels (TRL):
# {TRL_DEFINITIONS}

# Category Definitions:
# {json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

# Application Type Indicators:
# {format_application_indicators(APPLICATION_INDICATORS)}

# Instructions:
# 1. Determine if the document is relevant to heat pump innovation
# 2. Provide a summary (≤25 words) if relevant
# 3. Assign ONE primary category from the list above
# 4. Score 0-3 for:
#   # a) cost_reduction_potential (0=none, 3=breakthrough)
#   # b) efficiency_gain_potential (0=none, 3=breakthrough)
#    #c) circularity_score (ISO 59004 principles)
# 5. Assign TRL level (1-9) based on the maturity indicators in the text

# Return ONLY JSON with all seven fields requested.
#"""


In [17]:
# 4. Updated SYSTEM_MESSAGE keeping the formatted variables
SYSTEM_MESSAGE = f"""
You are a sustainable-heating technology analyst evaluating heat pump innovations.

CATEGORIES FOR CLASSIFICATION:
{json.dumps(CATEGORIES, indent=2)}

CATEGORY EXPLANATIONS:
{json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

Technology Readiness Levels (TRL):
{TRL_DEFINITIONS}

Application Type Indicators:
{format_application_indicators(APPLICATION_INDICATORS)}

SCORING CRITERIA:
Cost Reduction (0-3):
- 0: No cost reduction potential
- 1: Minor reduction (<10% cost savings)
- 2: Moderate reduction (10-30% cost savings)
- 3: Major reduction (>30% cost savings)

Efficiency Gain (0-3):
- 0: No efficiency improvement
- 1: Minor gain (<10% COP improvement)
- 2: Moderate gain (10-30% COP improvement)
- 3: Major gain (>30% COP improvement)

Circularity (0-3) - Based on ISO 59004 principles:
- 0: No circular economy principles
- 1: One principle (maintain/reuse/recycle/reduce/regenerate)
- 2: Two principles
- 3: Three or more principles

INSTRUCTIONS:
Analyze the patents and papers documents and provide a comprehensive assessment following the field definitions exactly.
For each scoring field, first determine if the potential exists (yes/no), then explain why, then assign the score.

Return ONLY valid JSON matching the specified fields.
"""

In [18]:
# 5 Structured output spec for batch_check with
# Updated FIELDS for better structure and constraints
FIELDS = [
    # Relevance and classification
    {"name": "is_relevant", "type": "str", 
     "description": "One-word answer: 'yes' if the text is about heat pump innovation or technology, otherwise 'no'."},
    {"name": "relevance_reason", "type": "str", 
     "description": "Short explanation (one sentence, ≤20 words) of why the text is relevant or not."},
    
    # Basic information
    {"name": "summary", "type": "str", 
     "description": "A brief summary (≤25 words) of the innovation or technology described."},
    {"name": "category", "type": "str", 
     "description": f"Select ONE primary category from: {', '.join(CATEGORIES.values())}"},
    
    # Application context
    {"name": "application_type", "type": "str", 
     "description": "Identify the application: 'domestic' (residential, <20kW), 'industrial' (commercial, >100kW), 'both', or 'unclear'."},
    {"name": "specific_applications", "type": "list[str]", 
     "description": "Specific use cases mentioned (e.g., 'space heating', 'water heating', 'process heat', 'district heating')."},
    
    # Technology readiness
    {"name": "trl_level", "type": "int", 
     "description": "Technology Readiness Level from 1-9: (1) Basic principles, (2) Concept formulated, (3) Proof of concept, (4) Lab validation, (5) Prototype tested, (6) Prototype in environment, (7) Pre-commercial demo, (8) First commercial, (9) Full commercial."},
    {"name": "trl_evidence", "type": "str", 
     "description": "Brief evidence supporting the TRL assessment (e.g., 'laboratory tests mentioned', 'commercial product available')."},
    
    # Cost reduction potential
    {"name": "has_cost_reduction_potential", "type": "str", 
     "description": "One-word answer: 'yes' if the innovation could reduce heat pump costs (upfront or running), otherwise 'no'."},
    {"name": "cost_reduction_reason", "type": "str", 
     "description": "Explanation of how this could reduce costs (e.g., 'cheaper materials', 'improved efficiency', 'reduced maintenance')."},
    {"name": "cost_reduction_score", "type": "int", 
     "description": "Score 0-3: (0) no cost reduction, (1) minor reduction (<10%), (2) moderate reduction (10-30%), (3) major reduction (>30%)."},
    
    # Efficiency improvement potential
    {"name": "has_efficiency_gain", "type": "str", 
     "description": "One-word answer: 'yes' if the innovation could improve heat pump efficiency/COP, otherwise 'no'."},
    {"name": "efficiency_gain_reason", "type": "str", 
     "description": "Explanation of efficiency improvements (e.g., 'better heat transfer', 'reduced losses', 'optimized cycle')."},
    {"name": "efficiency_gain_score", "type": "int", 
     "description": "Score 0-3: (0) no gain, (1) minor gain (<10% COP improvement), (2) moderate gain (10-30%), (3) major gain (>30%)."},
    
    # Circularity assessment
    {"name": "circularity_principles", "type": "list[str]", 
     "description": "Circular Economy principles present: select from 'maintain', 'reuse', 'recycle', 'reduce', 'regenerate', or 'none'."},
    {"name": "circularity_evidence", "type": "str", 
     "description": "Specific circular economy features mentioned (e.g., 'modular design', 'recycled materials', 'predictive maintenance')."},
    {"name": "circularity_score", "type": "int", 
     "description": "Score 0-3 based on number of Circular Economy principles: (0) none, (1) one principle, (2) two principles, (3) three or more."},
    
    # Technical details
    {"name": "key_innovations", "type": "list[str]", 
     "description": "List of specific technical innovations or improvements mentioned."},
    {"name": "materials_mentioned", "type": "list[str]", 
     "description": "Any specific materials or components mentioned (e.g., 'shape-memory alloy', 'R290 refrigerant')."},
]

In [19]:
# ## 5 – Structured output spec for batch_check
# FIELDS = [
#     {"name":"is_relevant","type":"str","description":"'yes' or 'no'"},
#     {"name":"summary","type":"str","description":"≤25 words"},
#     {"name":"category","type":"str","description":f"One of {list(CATEGORIES.values())}"},
#   {"name":"application_type","type":"str","description":"'domestic', 'industrial', 'both', or 'unclear'"}, #Domestic & Industrial
#     {"name":"cost_score","type":"int","description":"0-3"},
#     {"name":"efficiency_score","type":"int","description":"0-3"},
#     {"name":"circularity_score","type":"int","description":"0-3 (LLM view)"},
#     TRL addition
#     {"name":"trl_level","type":"int","description":"Technology Readiness Level 1-9"}
# ]



In [20]:
## 6 – Create ID→text dicts (full datasets)
paper_dict  = openalex_df.set_index("id")["title_abstract"].to_dict()
patent_dict = patents_df .set_index("publication_number")["title_abstract"].to_dict()


In [21]:
# Karlis's feedback: Distinguish Heat Pump Relevance
def prefilter_heat_pump_relevance(text_dict, sample_size=100):
    """
    First stage: Check if documents are actually about heat pumps
    """
    # Create a small sample for testing
    sample_ids = list(text_dict.keys())[:sample_size]
    sample_dict = {id: text_dict[id] for id in sample_ids}
    
    # Special system message for relevance checking
    RELEVANCE_SYSTEM = """
    You are evaluating if a document is PRIMARILY about heat pump technology.
    
    Heat pumps are devices that transfer heat from one place to another using a refrigeration cycle.
    They are used for heating, cooling, or both.
    
    Return JSON with:
    - "is_heat_pump": "yes" if primarily about heat pumps, "no" if just mentions pumps or is unrelated
    - "confidence": 0-1 (how confident you are)
    - "reason": brief explanation
    """
    
    relevance_fields = [
        {"name": "is_heat_pump", "type": "str", "description": "'yes' or 'no'"},
        {"name": "confidence", "type": "float", "description": "0-1 confidence score"},
        {"name": "reason", "type": "str", "description": "Brief explanation"}
    ]
    
    # Run relevance check
    relevance_proc = batch_check.LLMProcessor(
        model_name="gpt-4o-mini", temperature=0,
        system_message=RELEVANCE_SYSTEM, 
        session_name="heat_pump_relevance",
        output_fields=relevance_fields, 
        output_path=str(OUTPUT_DIR/"relevance_check.jsonl")
    )
    
    return relevance_proc.run(sample_dict, batch_size=30, sleep_time=0.5)

In [22]:
# ## 7 – Run GPT in batches *and* post-process ISO score
# OUTPUT_DIR = PROJECT_DIR / "outputs"; 
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# def run_and_enrich(data, name, path, batch=30):
#     proc = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini", temperature=0,
#         system_message=SYSTEM_MESSAGE, session_name=name,
#         output_fields=FIELDS, output_path=str(path)
#     )
#     proc.run(data, batch_size=batch, sleep_time=0.5)

#     return


# pat_out = run_and_enrich(
#     {k:v for k,v in patent_dict.items() if any(w in v for w in KW2CAT)},
#     "hp_patents_full", OUTPUT_DIR/"patents4_classified.jsonl"
# )

# pap_out = run_and_enrich(
#     paper_dict, "hp_papers_full", OUTPUT_DIR/"papers4_classified.jsonl"
# )


In [30]:
## 7 – Two-Stage Processing with GPT

OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1: Classification
def stage1_classify(data_dict, dataset_name):
    """Stage 1: Basic classification and relevance"""
    
    CLASSIFY_SYSTEM = f"""
You are classifying heat pump technology documents.

CATEGORIES:
{json.dumps(CATEGORIES, indent=2)}

CATEGORY EXPLANATIONS:
{json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

INSTRUCTIONS:
1. Determine if the document is PRIMARILY about heat pump innovation or technology
2. If relevant, assign the most appropriate category
3. Provide a brief summary of the innovation
4. List key technical innovations mentioned

Return JSON with these fields only.
"""
    
    classify_fields = [
        {"name": "is_relevant", "type": "str", 
         "description": "'yes' if primarily about heat pump innovation, 'no' otherwise"},
        {"name": "relevance_reason", "type": "str", 
         "description": "Brief explanation (max 20 words) of relevance decision"},
        {"name": "category", "type": "str", 
         "description": f"Primary category from: {', '.join(CATEGORIES.values())}. Use 'N/A' if not relevant."},
        {"name": "summary", "type": "str", 
         "description": "Brief summary (≤25 words) of the innovation. Use 'N/A' if not relevant."},
        {"name": "key_innovations", "type": "list[str]", 
         "description": "List of specific technical features mentioned"},
    ]
    
    proc1 = batch_check.LLMProcessor(
        model_name="gpt-4o-mini",
        temperature=0,
        system_message=CLASSIFY_SYSTEM,
        session_name=f"{dataset_name}_stage1_classify",
        output_fields=classify_fields,
        output_path=str(OUTPUT_DIR / f"{dataset_name}_stage1_classified_v2.jsonl")
    )
    
    proc1.run(data_dict, batch_size=30, sleep_time=0.5)
    
    # Read results and return as dictionary
    results_df = pd.read_json(OUTPUT_DIR / f"{dataset_name}_stage1_classified.jsonl", lines=True)
    return results_df.set_index('_id').to_dict('index')

In [24]:
# ## 7 – Two-Stage Processing with GPT

# OUTPUT_DIR = PROJECT_DIR / "outputs"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # Stage 1: Classification
# def stage1_classify(data_dict, dataset_name):
#     """Stage 1: Basic classification and relevance"""
    
#     CLASSIFY_SYSTEM = f"""
# You are classifying heat pump technology documents.

# CATEGORIES:
# {json.dumps(CATEGORIES, indent=2)}

# CATEGORY EXPLANATIONS:
# {json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

# INSTRUCTIONS:
# 1. Determine if the document is PRIMARILY about heat pump innovation or technology
# 2. If relevant, assign the most appropriate category
# 3. Provide a brief summary of the innovation
# 4. List key technical innovations mentioned

# Return JSON with these fields only.
# """
    
#     classify_fields = [
#         {"name": "is_relevant", "type": "str", 
#          "description": "'yes' if primarily about heat pump innovation, 'no' otherwise"},
#         {"name": "relevance_reason", "type": "str", 
#          "description": "Brief explanation (max 20 words) of relevance decision"},
#         {"name": "category", "type": "str", 
#          "description": f"Primary category from: {', '.join(CATEGORIES.values())}. Use 'N/A' if not relevant."},
#         {"name": "summary", "type": "str", 
#          "description": "Brief summary (≤25 words) of the innovation. Use 'N/A' if not relevant."},
#         {"name": "key_innovations", "type": "list[str]", 
#          "description": "List of specific technical features mentioned"},
#     ]
    
#     proc1 = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini",
#         temperature=0,
#         system_message=CLASSIFY_SYSTEM,
#         session_name=f"{dataset_name}_stage1_classify",
#         output_fields=classify_fields,
#         output_path=str(OUTPUT_DIR / f"{dataset_name}_stage1_classified.jsonl")
#     )
    
#     proc1.run(data_dict, batch_size=30, sleep_time=0.5)
    
#     # Read results and return as dictionary
#     results_df = pd.read_json(OUTPUT_DIR / f"{dataset_name}_stage1_classified.jsonl", lines=True)
#     return results_df.set_index('_id').to_dict('index')

# # Stage 2: Detailed Scoring
# def stage2_score(classified_results, original_texts, dataset_name):
#     """Stage 2: Detailed scoring of relevant documents only"""
    
#     SCORE_SYSTEM = f"""
# You are scoring heat pump innovations for their potential impact.

# Technology Readiness Levels (TRL):
# {TRL_DEFINITIONS}

# Application Type Indicators:
# {format_application_indicators(APPLICATION_INDICATORS)}

# ISO 59004 Circularity Principles:
# - Maintain: predictive maintenance, condition monitoring, service models
# - Reuse: remanufacturing, refurbishment, modular replacement
# - Recycle: material recovery, closed-loop systems
# - Reduce: efficient manufacturing, minimal waste
# - Regenerate: bio-based materials, renewable resources

# SCORING CRITERIA:
# Cost Reduction (0-3):
# - 0: No cost reduction potential
# - 1: Minor reduction (<10% cost savings)
# - 2: Moderate reduction (10-30% cost savings)
# - 3: Major reduction (>30% cost savings)

# Efficiency Gain (0-3):
# - 0: No efficiency improvement
# - 1: Minor gain (<10% COP improvement)
# - 2: Moderate gain (10-30% COP improvement)
# - 3: Major gain (>30% COP improvement)

# INSTRUCTIONS:
# Based on the document, assess the innovation's potential and maturity.
# Consider both explicit claims and reasonable inferences from the technology described.
# """
    
#     score_fields = [
#         # Application context
#         {"name": "application_type", "type": "str", 
#          "description": "'domestic', 'industrial', 'both', or 'unclear'"},
#         {"name": "specific_applications", "type": "list[str]", 
#          "description": "Specific use cases (e.g., 'space heating', 'water heating')"},
        
#         # Technology readiness
#         {"name": "trl_level", "type": "int", 
#          "description": "TRL 1-9 based on development stage described"},
#         {"name": "trl_evidence", "type": "str", 
#          "description": "Evidence supporting TRL assessment"},
        
#         # Cost potential
#         {"name": "has_cost_reduction_potential", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "cost_reduction_reason", "type": "str", 
#          "description": "How it reduces costs (if applicable)"},
#         {"name": "cost_reduction_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
        
#         # Efficiency potential
#         {"name": "has_efficiency_gain", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "efficiency_gain_reason", "type": "str", 
#          "description": "How it improves efficiency (if applicable)"},
#         {"name": "efficiency_gain_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
        
#         # Circularity
#         {"name": "circularity_principles", "type": "list[str]", 
#          "description": "Which principles apply: 'maintain', 'reuse', 'recycle', 'reduce', 'regenerate'"},
#         {"name": "circularity_evidence", "type": "str", 
#          "description": "Specific circular features mentioned"},
#         {"name": "circularity_score", "type": "int", 
#          "description": "0-3 based on number of principles"},
        
#         # Materials
#         {"name": "materials_mentioned", "type": "list[str]", 
#          "description": "Specific materials or components mentioned"},
#     ]
    
#     # Filter only relevant documents
#     relevant_ids = {doc_id: results for doc_id, results in classified_results.items() 
#                    if results.get('is_relevant') == 'yes'}
    
#     print(f"Stage 2: Scoring {len(relevant_ids)} relevant documents out of {len(classified_results)} total")
    
#     # Get original texts for relevant documents
#     relevant_texts = {doc_id: original_texts[doc_id] for doc_id in relevant_ids.keys()}
    
#     proc2 = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini",
#         temperature=0,
#         system_message=SCORE_SYSTEM,
#         session_name=f"{dataset_name}_stage2_score",
#         output_fields=score_fields,
#         output_path=str(OUTPUT_DIR / f"{dataset_name}_stage2_scored.jsonl")
#     )
    
#     proc2.run(relevant_texts, batch_size=30, sleep_time=0.5)
    
#     # Read scoring results
#     scores_df = pd.read_json(OUTPUT_DIR / f"{dataset_name}_stage2_scored.jsonl", lines=True)
#     return scores_df.set_index('_id').to_dict('index')

# # Combine results function
# def combine_results(stage1_results, stage2_results, dataset_name):
#     """Combine results from both stages into final output"""
    
#     combined = {}
    
#     for doc_id, stage1 in stage1_results.items():
#         if stage1.get('is_relevant') == 'yes' and doc_id in stage2_results:
#             # Combine relevant documents with both stages
#             combined[doc_id] = {**stage1, **stage2_results[doc_id]}
#         else:
#             # Keep non-relevant documents with stage 1 info only
#             combined[doc_id] = stage1
#             # Add empty fields for consistency
#             combined[doc_id].update({
#                 'application_type': 'N/A',
#                 'specific_applications': [],
#                 'trl_level': 0,
#                 'trl_evidence': 'N/A',
#                 'has_cost_reduction_potential': 'no',
#                 'cost_reduction_reason': 'N/A',
#                 'cost_reduction_score': 0,
#                 'has_efficiency_gain': 'no',
#                 'efficiency_gain_reason': 'N/A',
#                 'efficiency_gain_score': 0,
#                 'circularity_principles': [],
#                 'circularity_evidence': 'N/A',
#                 'circularity_score': 0,
#                 'materials_mentioned': []
#             })
    
#     # Save combined results
#     combined_df = pd.DataFrame.from_dict(combined, orient='index')
#     combined_df.index.name = '_id'
#     combined_df.to_json(
#         OUTPUT_DIR / f"{dataset_name}_final_combined.jsonl",
#         orient='records',
#         lines=True
#     )
    
#     return combined_df

# # Main processing function
# def process_dataset_two_stage(text_dict, dataset_name):
#     """Process a dataset through both stages"""
    
#     print(f"\nProcessing {dataset_name} dataset...")
#     print(f"Total documents: {len(text_dict)}")
    
#     # Stage 1
#     print("\nStage 1: Classification and relevance...")
#     stage1_results = stage1_classify(text_dict, dataset_name)
    
#     # Stage 2
#     print("\nStage 2: Detailed scoring...")
#     stage2_results = stage2_score(stage1_results, text_dict, dataset_name)
    
#     # Combine
#     print("\nCombining results...")
#     final_results = combine_results(stage1_results, stage2_results, dataset_name)
    
#     # Print summary statistics
#     print(f"\nSummary for {dataset_name}:")
#     print(f"Total documents: {len(final_results)}")
#     print(f"Relevant documents: {sum(1 for _, row in final_results.iterrows() if row['is_relevant'] == 'yes')}")
#     print(f"Average cost score: {final_results['cost_reduction_score'].mean():.2f}")
#     print(f"Average efficiency score: {final_results['efficiency_gain_score'].mean():.2f}")
#     print(f"Average circularity score: {final_results['circularity_score'].mean():.2f}")
    
#     return final_results

# # Run the two-stage process
# # For patents - with keyword filtering
# patent_subset = {k: v for k, v in patent_dict.items() if contains_keyword(v)}
# print(f"Patents after keyword filtering: {len(patent_subset)} out of {len(patent_dict)}")
# patent_results = process_dataset_two_stage(patent_subset, "patents_v5")

# # For papers - no keyword filtering
# paper_results = process_dataset_two_stage(paper_dict, "papers_v5")

# # Optional: Add ISO heuristic score for comparison
# def add_iso_heuristic_score(results_df, text_dict):
#     """Add ISO heuristic score for comparison with LLM scoring"""
#     results_df['iso_heuristic_score'] = results_df.index.map(
#         lambda x: iso_score(text_dict.get(x, ''))
#     )
#     return results_df

# patent_results = add_iso_heuristic_score(patent_results, patent_dict)
# paper_results = add_iso_heuristic_score(paper_results, paper_dict)

# print("\nProcessing complete!")

In [25]:
# ## 7 – Two-Stage Processing with GPT

# OUTPUT_DIR = PROJECT_DIR / "outputs"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # Stage 1: Classification
# def stage1_classify(data_dict, dataset_name):
#     """Stage 1: Basic classification and relevance"""
    
#     CLASSIFY_SYSTEM = f"""
# You are classifying heat pump technology documents.

# CATEGORIES:
# {json.dumps(CATEGORIES, indent=2)}

# CATEGORY EXPLANATIONS:
# {json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

# INSTRUCTIONS:
# 1. Determine if the document is PRIMARILY about heat pump innovation or technology
# 2. If relevant, assign the most appropriate category
# 3. Provide a brief summary of the innovation
# 4. List key technical innovations mentioned

# Return JSON with these fields only.
# """
    
#     classify_fields = [
#         {"name": "is_relevant", "type": "str", 
#          "description": "'yes' if primarily about heat pump innovation, 'no' otherwise"},
#         {"name": "relevance_reason", "type": "str", 
#          "description": "Brief explanation (max 20 words) of relevance decision"},
#         {"name": "category", "type": "str", 
#          "description": f"Primary category from: {', '.join(CATEGORIES.values())}. Use 'N/A' if not relevant."},
#         {"name": "summary", "type": "str", 
#          "description": "Brief summary (≤25 words) of the innovation. Use 'N/A' if not relevant."},
#         {"name": "key_innovations", "type": "list[str]", 
#          "description": "List of specific technical features mentioned"},
#     ]
    
#     output_path = OUTPUT_DIR / f"{dataset_name}_stage1_classified.jsonl"
    
#     proc1 = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini",
#         temperature=0,
#         system_message=CLASSIFY_SYSTEM,
#         session_name=f"{dataset_name}_stage1_classify",
#         output_fields=classify_fields,
#         output_path=str(output_path)
#     )
    
#     # Run the processor
#     proc1.run(data_dict, batch_size=30, sleep_time=0.5)
    
#     # Wait a moment for the file to be written
#     import time
#     time.sleep(2)
    
#     # Check if the output file exists and has content
#     if output_path.exists() and output_path.stat().st_size > 0:
#         # Read results and return as dictionary
#         results_df = pd.read_json(output_path, lines=True)
#         if '_id' in results_df.columns:
#             return results_df.set_index('_id').to_dict('index')
#         else:
#             # If _id column doesn't exist, create it from index
#             results_df['_id'] = results_df.index
#             return results_df.set_index('_id').to_dict('index')
#     else:
#         print(f"Warning: No results found in {output_path}")
#         return {}

# # Stage 2: Detailed Scoring
# def stage2_score(classified_results, original_texts, dataset_name):
#     """Stage 2: Detailed scoring of relevant documents only"""
    
#     # Filter only relevant documents
#     relevant_ids = {doc_id: results for doc_id, results in classified_results.items() 
#                    if results.get('is_relevant') == 'yes'}
    
#     if not relevant_ids:
#         print("No relevant documents found in Stage 1")
#         return {}
    
#     SCORE_SYSTEM = f"""
# You are scoring heat pump innovations for their potential impact.

# Technology Readiness Levels (TRL):
# {TRL_DEFINITIONS}

# Application Type Indicators:
# {format_application_indicators(APPLICATION_INDICATORS)}

# ISO 59004 Circularity Principles:
# - Maintain: predictive maintenance, condition monitoring, service models
# - Reuse: remanufacturing, refurbishment, modular replacement
# - Recycle: material recovery, closed-loop systems
# - Reduce: efficient manufacturing, minimal waste
# - Regenerate: bio-based materials, renewable resources

# SCORING CRITERIA:
# Cost Reduction (0-3):
# - 0: No cost reduction potential
# - 1: Minor reduction (<10% cost savings)
# - 2: Moderate reduction (10-30% cost savings)
# - 3: Major reduction (>30% cost savings)

# Efficiency Gain (0-3):
# - 0: No efficiency improvement
# - 1: Minor gain (<10% COP improvement)
# - 2: Moderate gain (10-30% COP improvement)
# - 3: Major gain (>30% COP improvement)

# INSTRUCTIONS:
# Based on the document, assess the innovation's potential and maturity.
# Consider both explicit claims and reasonable inferences from the technology described.
# """
    
#     score_fields = [
#         # Application context
#         {"name": "application_type", "type": "str", 
#          "description": "'domestic', 'industrial', 'both', or 'unclear'"},
#         {"name": "specific_applications", "type": "list[str]", 
#          "description": "Specific use cases (e.g., 'space heating', 'water heating')"},
        
#         # Technology readiness
#         {"name": "trl_level", "type": "int", 
#          "description": "TRL 1-9 based on development stage described"},
#         {"name": "trl_evidence", "type": "str", 
#          "description": "Evidence supporting TRL assessment"},
        
#         # Cost potential
#         {"name": "has_cost_reduction_potential", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "cost_reduction_reason", "type": "str", 
#          "description": "How it reduces costs (if applicable)"},
#         {"name": "cost_reduction_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
        
#         # Efficiency potential
#         {"name": "has_efficiency_gain", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "efficiency_gain_reason", "type": "str", 
#          "description": "How it improves efficiency (if applicable)"},
#         {"name": "efficiency_gain_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
        
#         # Circularity
#         {"name": "circularity_principles", "type": "list[str]", 
#          "description": "Which principles apply: 'maintain', 'reuse', 'recycle', 'reduce', 'regenerate'"},
#         {"name": "circularity_evidence", "type": "str", 
#          "description": "Specific circular features mentioned"},
#         {"name": "circularity_score", "type": "int", 
#          "description": "0-3 based on number of principles"},
        
#         # Materials
#         {"name": "materials_mentioned", "type": "list[str]", 
#          "description": "Specific materials or components mentioned"},
#     ]
    
#     print(f"Stage 2: Scoring {len(relevant_ids)} relevant documents out of {len(classified_results)} total")
    
#     # Get original texts for relevant documents
#     relevant_texts = {doc_id: original_texts[doc_id] for doc_id in relevant_ids.keys()}
    
#     output_path = OUTPUT_DIR / f"{dataset_name}_stage2_scored.jsonl"
    
#     proc2 = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini",
#         temperature=0,
#         system_message=SCORE_SYSTEM,
#         session_name=f"{dataset_name}_stage2_score",
#         output_fields=score_fields,
#         output_path=str(output_path)
#     )
    
#     proc2.run(relevant_texts, batch_size=30, sleep_time=0.5)
    
#     # Wait for file to be written
#     time.sleep(2)
    
#     # Check if the output file exists and has content
#     if output_path.exists() and output_path.stat().st_size > 0:
#         # Read scoring results
#         scores_df = pd.read_json(output_path, lines=True)
#         if '_id' in scores_df.columns:
#             return scores_df.set_index('_id').to_dict('index')
#         else:
#             scores_df['_id'] = scores_df.index
#             return scores_df.set_index('_id').to_dict('index')
#     else:
#         print(f"Warning: No results found in {output_path}")
#         return {}

# # Combine results function
# def combine_results(stage1_results, stage2_results, dataset_name):
#     """Combine results from both stages into final output"""
    
#     if not stage1_results:
#         print("No Stage 1 results to combine")
#         return pd.DataFrame()
    
#     combined = {}
    
#     for doc_id, stage1 in stage1_results.items():
#         if stage1.get('is_relevant') == 'yes' and doc_id in stage2_results:
#             # Combine relevant documents with both stages
#             combined[doc_id] = {**stage1, **stage2_results[doc_id]}
#         else:
#             # Keep non-relevant documents with stage 1 info only
#             combined[doc_id] = stage1
#             # Add empty fields for consistency
#             combined[doc_id].update({
#                 'application_type': 'N/A',
#                 'specific_applications': [],
#                 'trl_level': 0,
#                 'trl_evidence': 'N/A',
#                 'has_cost_reduction_potential': 'no',
#                 'cost_reduction_reason': 'N/A',
#                 'cost_reduction_score': 0,
#                 'has_efficiency_gain': 'no',
#                 'efficiency_gain_reason': 'N/A',
#                 'efficiency_gain_score': 0,
#                 'circularity_principles': [],
#                 'circularity_evidence': 'N/A',
#                 'circularity_score': 0,
#                 'materials_mentioned': []
#             })
    
#     # Save combined results
#     combined_df = pd.DataFrame.from_dict(combined, orient='index')
#     combined_df.index.name = '_id'
    
#     # Save to file
#     output_path = OUTPUT_DIR / f"{dataset_name}_final_combined.jsonl"
#     combined_df.reset_index().to_json(
#         output_path,
#         orient='records',
#         lines=True
#     )
    
#     return combined_df

# # Main processing function
# def process_dataset_two_stage(text_dict, dataset_name):
#     """Process a dataset through both stages"""
    
#     print(f"\nProcessing {dataset_name} dataset...")
#     print(f"Total documents: {len(text_dict)}")
    
#     # Stage 1
#     print("\nStage 1: Classification and relevance...")
#     stage1_results = stage1_classify(text_dict, dataset_name)
    
#     if not stage1_results:
#         print(f"No results from Stage 1 for {dataset_name}")
#         return pd.DataFrame()
    
#     # Stage 2
#     print("\nStage 2: Detailed scoring...")
#     stage2_results = stage2_score(stage1_results, text_dict, dataset_name)
    
#     # Combine
#     print("\nCombining results...")
#     final_results = combine_results(stage1_results, stage2_results, dataset_name)
    
#     if len(final_results) > 0:
#         # Print summary statistics
#         print(f"\nSummary for {dataset_name}:")
#         print(f"Total documents: {len(final_results)}")
#         print(f"Relevant documents: {sum(1 for _, row in final_results.iterrows() if row.get('is_relevant') == 'yes')}")
        
#         # Only calculate means if there are numeric values
#         if 'cost_reduction_score' in final_results.columns:
#             print(f"Average cost score: {final_results['cost_reduction_score'].mean():.2f}")
#         if 'efficiency_gain_score' in final_results.columns:
#             print(f"Average efficiency score: {final_results['efficiency_gain_score'].mean():.2f}")
#         if 'circularity_score' in final_results.columns:
#             print(f"Average circularity score: {final_results['circularity_score'].mean():.2f}")
    
#     return final_results

# # Optional: Test with a small subset first
# def test_with_subset(text_dict, n=10):
#     """Test the pipeline with a small subset of documents"""
#     subset = dict(list(text_dict.items())[:n])
#     return process_dataset_two_stage(subset, "test_subset")

# # Uncomment to test with a small subset first:
# # test_results = test_with_subset(patent_dict, n=5)

# # Run the two-stage process
# # For patents - with keyword filtering
# patent_subset = {k: v for k, v in patent_dict.items() if contains_keyword(v)}
# print(f"Patents after keyword filtering: {len(patent_subset)} out of {len(patent_dict)}")

# # Process in smaller batches if needed
# if len(patent_subset) > 1000:
#     print("Processing patents in batches...")
#     # Process first 1000 patents
#     patent_subset_small = dict(list(patent_subset.items())[:1000])
#     patent_results = process_dataset_two_stage(patent_subset_small, "patents_v5_batch1")
# else:
#     patent_results = process_dataset_two_stage(patent_subset, "patents_v5")

# # For papers - process smaller subset
# print("\nProcessing papers...")
# if len(paper_dict) > 500:
#     paper_subset = dict(list(paper_dict.items())[:500])
#     paper_results = process_dataset_two_stage(paper_subset, "papers_v5_subset")
# else:
#     paper_results = process_dataset_two_stage(paper_dict, "papers_v5")

# # Optional: Add ISO heuristic score for comparison
# def add_iso_heuristic_score(results_df, text_dict):
#     """Add ISO heuristic score for comparison with LLM scoring"""
#     if len(results_df) > 0:
#         results_df['iso_heuristic_score'] = results_df.index.map(
#             lambda x: iso_score(text_dict.get(x, ''))
#         )
#     return results_df

# if len(patent_results) > 0:
#     patent_results = add_iso_heuristic_score(patent_results, patent_dict)
    
# if len(paper_results) > 0:
#     paper_results = add_iso_heuristic_score(paper_results, paper_dict)

# print("\nProcessing complete!")

# # Show sample results
# if len(patent_results) > 0:
#     print("\nSample patent results:")
#     print(patent_results.head())

# if len(paper_results) > 0:
#     print("\nSample paper results:")
#     print(paper_results.head())

In [26]:
# ## 7 – Two-Stage Processing with GPT

# import time
# import os

# OUTPUT_DIR = PROJECT_DIR / "outputs"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # Helper functions
# def wait_for_file(filepath, timeout=300, check_interval=5):
#     """Wait for a file to be created and have content"""
#     start_time = time.time()
#     while time.time() - start_time < timeout:
#         if filepath.exists() and filepath.stat().st_size > 0:
#             # Check if file is still being written by comparing sizes
#             size1 = filepath.stat().st_size
#             time.sleep(2)
#             size2 = filepath.stat().st_size
#             if size1 == size2:  # File size is stable
#                 return True
#         time.sleep(check_interval)
#     return False

# def count_lines_in_file(filepath):
#     """Count lines in a JSONL file"""
#     if filepath.exists():
#         with open(filepath, 'r') as f:
#             return sum(1 for _ in f)
#     return 0

# # Stage 1: Classification
# def stage1_classify(data_dict, dataset_name):
#     """Stage 1: Basic classification and relevance"""
    
#     CLASSIFY_SYSTEM = f"""
# You are classifying heat pump technology documents.

# CATEGORIES:
# {json.dumps(CATEGORIES, indent=2)}

# CATEGORY EXPLANATIONS:
# {json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

# INSTRUCTIONS:
# 1. Determine if the document is PRIMARILY about heat pump innovation or technology
# 2. If relevant, assign the most appropriate category
# 3. Provide a brief summary of the innovation
# 4. List key technical innovations mentioned

# Return JSON with these fields only.
# """
    
#     classify_fields = [
#         {"name": "is_relevant", "type": "str", 
#          "description": "'yes' if primarily about heat pump innovation, 'no' otherwise"},
#         {"name": "relevance_reason", "type": "str", 
#          "description": "Brief explanation (max 20 words) of relevance decision"},
#         {"name": "category", "type": "str", 
#          "description": f"Primary category from: {', '.join(CATEGORIES.values())}. Use 'N/A' if not relevant."},
#         {"name": "summary", "type": "str", 
#          "description": "Brief summary (≤25 words) of the innovation. Use 'N/A' if not relevant."},
#         {"name": "key_innovations", "type": "list[str]", 
#          "description": "List of specific technical features mentioned"},
#     ]
    
#     output_path = OUTPUT_DIR / f"{dataset_name}_stage1_classified.jsonl"
    
#     # Remove existing file if it exists
#     if output_path.exists():
#         output_path.unlink()
    
#     proc1 = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini",
#         temperature=0,
#         system_message=CLASSIFY_SYSTEM,
#         session_name=f"{dataset_name}_stage1_classify",
#         output_fields=classify_fields,
#         output_path=str(output_path)
#     )
    
#     # Calculate expected number of batches
#     batch_size = 30
#     expected_batches = (len(data_dict) + batch_size - 1) // batch_size
#     print(f"Expected number of batches: {expected_batches}")
    
#     # Run the processor
#     proc1.run(data_dict, batch_size=batch_size, sleep_time=0.5)
    
#     # Wait for processing to complete
#     print(f"Waiting for Stage 1 processing to complete...")
#     print(f"Output file: {output_path}")
    
#     # Monitor progress
#     last_count = 0
#     no_change_count = 0
#     max_wait_cycles = 60  # Maximum 5 minutes (60 * 5 seconds)
#     wait_cycles = 0
    
#     while wait_cycles < max_wait_cycles:
#         if output_path.exists():
#             current_count = count_lines_in_file(output_path)
#             if current_count > last_count:
#                 print(f"Processed {current_count}/{len(data_dict)} documents...")
#                 last_count = current_count
#                 no_change_count = 0
#             else:
#                 no_change_count += 1
            
#             # If we've processed all documents or no change for a while
#             if current_count >= len(data_dict) or no_change_count > 10:
#                 break
#         else:
#             print("Waiting for output file to be created...")
        
#         time.sleep(5)
#         wait_cycles += 1
    
#     # Read results if file exists
#     if output_path.exists() and output_path.stat().st_size > 0:
#         try:
#             results_df = pd.read_json(output_path, lines=True)
#             print(f"Successfully read {len(results_df)} results from Stage 1")
            
#             # Check for _id column and create if needed
#             if '_id' not in results_df.columns:
#                 # The batch_check processor might use a different ID column
#                 if 'id' in results_df.columns:
#                     results_df['_id'] = results_df['id']
#                 elif len(results_df) == len(data_dict):
#                     results_df['_id'] = list(data_dict.keys())
#                 else:
#                     print("Warning: Results count doesn't match input count")
#                     results_df['_id'] = results_df.index
            
#             return results_df.set_index('_id').to_dict('index')
#         except Exception as e:
#             print(f"Error reading results: {e}")
#             return {}
#     else:
#         print(f"No results found in {output_path}")
#         return {}

# # Stage 2: Detailed Scoring
# def stage2_score(classified_results, original_texts, dataset_name):
#     """Stage 2: Detailed scoring of relevant documents only"""
    
#     # Filter only relevant documents
#     relevant_ids = {doc_id: results for doc_id, results in classified_results.items() 
#                    if results.get('is_relevant') == 'yes'}
    
#     if not relevant_ids:
#         print("No relevant documents found in Stage 1")
#         return {}
    
#     SCORE_SYSTEM = f"""
# You are scoring heat pump innovations for their potential impact.

# Technology Readiness Levels (TRL):
# {TRL_DEFINITIONS}

# Application Type Indicators:
# {format_application_indicators(APPLICATION_INDICATORS)}

# ISO 59004 Circularity Principles:
# - Maintain: predictive maintenance, condition monitoring, service models
# - Reuse: remanufacturing, refurbishment, modular replacement
# - Recycle: material recovery, closed-loop systems
# - Reduce: efficient manufacturing, minimal waste
# - Regenerate: bio-based materials, renewable resources

# SCORING CRITERIA:
# Cost Reduction (0-3):
# - 0: No cost reduction potential
# - 1: Minor reduction (<10% cost savings)
# - 2: Moderate reduction (10-30% cost savings)
# - 3: Major reduction (>30% cost savings)

# Efficiency Gain (0-3):
# - 0: No efficiency improvement
# - 1: Minor gain (<10% COP improvement)
# - 2: Moderate gain (10-30% COP improvement)
# - 3: Major gain (>30% COP improvement)

# INSTRUCTIONS:
# Based on the document, assess the innovation's potential and maturity.
# Consider both explicit claims and reasonable inferences from the technology described.
# """
    
#     score_fields = [
#         # Application context
#         {"name": "application_type", "type": "str", 
#          "description": "'domestic', 'industrial', 'both', or 'unclear'"},
#         {"name": "specific_applications", "type": "list[str]", 
#          "description": "Specific use cases (e.g., 'space heating', 'water heating')"},
        
#         # Technology readiness
#         {"name": "trl_level", "type": "int", 
#          "description": "TRL 1-9 based on development stage described"},
#         {"name": "trl_evidence", "type": "str", 
#          "description": "Evidence supporting TRL assessment"},
        
#         # Cost potential
#         {"name": "has_cost_reduction_potential", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "cost_reduction_reason", "type": "str", 
#          "description": "How it reduces costs (if applicable)"},
#         {"name": "cost_reduction_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
        
#         # Efficiency potential
#         {"name": "has_efficiency_gain", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "efficiency_gain_reason", "type": "str", 
#          "description": "How it improves efficiency (if applicable)"},
#         {"name": "efficiency_gain_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
        
#         # Circularity
#         {"name": "circularity_principles", "type": "list[str]", 
#          "description": "Which principles apply: 'maintain', 'reuse', 'recycle', 'reduce', 'regenerate'"},
#         {"name": "circularity_evidence", "type": "str", 
#          "description": "Specific circular features mentioned"},
#         {"name": "circularity_score", "type": "int", 
#          "description": "0-3 based on number of principles"},
        
#         # Materials
#         {"name": "materials_mentioned", "type": "list[str]", 
#          "description": "Specific materials or components mentioned"},
#     ]
    
#     print(f"Stage 2: Scoring {len(relevant_ids)} relevant documents out of {len(classified_results)} total")
    
#     # Get original texts for relevant documents
#     relevant_texts = {doc_id: original_texts[doc_id] for doc_id in relevant_ids.keys()}
    
#     output_path = OUTPUT_DIR / f"{dataset_name}_stage2_scored.jsonl"
    
#     # Remove existing file if it exists
#     if output_path.exists():
#         output_path.unlink()
    
#     proc2 = batch_check.LLMProcessor(
#         model_name="gpt-4o-mini",
#         temperature=0,
#         system_message=SCORE_SYSTEM,
#         session_name=f"{dataset_name}_stage2_score",
#         output_fields=score_fields,
#         output_path=str(output_path)
#     )
    
#     proc2.run(relevant_texts, batch_size=30, sleep_time=0.5)
    
#     # Wait for processing
#     print("Waiting for Stage 2 processing to complete...")
#     last_count = 0
#     no_change_count = 0
#     max_wait_cycles = 60
#     wait_cycles = 0
    
#     while wait_cycles < max_wait_cycles:
#         if output_path.exists():
#             current_count = count_lines_in_file(output_path)
#             if current_count > last_count:
#                 print(f"Processed {current_count}/{len(relevant_texts)} documents...")
#                 last_count = current_count
#                 no_change_count = 0
#             else:
#                 no_change_count += 1
            
#             if current_count >= len(relevant_texts) or no_change_count > 10:
#                 break
        
#         time.sleep(5)
#         wait_cycles += 1
    
#     # Read results
#     if output_path.exists() and output_path.stat().st_size > 0:
#         try:
#             scores_df = pd.read_json(output_path, lines=True)
#             print(f"Successfully read {len(scores_df)} results from Stage 2")
            
#             if '_id' not in scores_df.columns:
#                 if 'id' in scores_df.columns:
#                     scores_df['_id'] = scores_df['id']
#                 else:
#                     scores_df['_id'] = scores_df.index
                    
#             return scores_df.set_index('_id').to_dict('index')
#         except Exception as e:
#             print(f"Error reading stage 2 results: {e}")
#             return {}
#     else:
#         print(f"No results found in {output_path}")
#         return {}

# # Combine results function
# def combine_results(stage1_results, stage2_results, dataset_name):
#     """Combine results from both stages into final output"""
    
#     if not stage1_results:
#         print("No Stage 1 results to combine")
#         return pd.DataFrame()
    
#     combined = {}
    
#     for doc_id, stage1 in stage1_results.items():
#         if stage1.get('is_relevant') == 'yes' and doc_id in stage2_results:
#             # Combine relevant documents with both stages
#             combined[doc_id] = {**stage1, **stage2_results[doc_id]}
#         else:
#             # Keep non-relevant documents with stage 1 info only
#             combined[doc_id] = stage1
#             # Add empty fields for consistency
#             combined[doc_id].update({
#                 'application_type': 'N/A',
#                 'specific_applications': [],
#                 'trl_level': 0,
#                 'trl_evidence': 'N/A',
#                 'has_cost_reduction_potential': 'no',
#                 'cost_reduction_reason': 'N/A',
#                 'cost_reduction_score': 0,
#                 'has_efficiency_gain': 'no',
#                 'efficiency_gain_reason': 'N/A',
#                 'efficiency_gain_score': 0,
#                 'circularity_principles': [],
#                 'circularity_evidence': 'N/A',
#                 'circularity_score': 0,
#                 'materials_mentioned': []
#             })
    
#     # Save combined results
#     combined_df = pd.DataFrame.from_dict(combined, orient='index')
#     combined_df.index.name = '_id'
    
#     # Save to file
#     output_path = OUTPUT_DIR / f"{dataset_name}_final_combined.jsonl"
#     combined_df.reset_index().to_json(
#         output_path,
#         orient='records',
#         lines=True
#     )
    
#     print(f"Saved combined results to {output_path}")
    
#     return combined_df

# # Main processing function
# def process_dataset_two_stage(text_dict, dataset_name):
#     """Process a dataset through both stages"""
    
#     print(f"\nProcessing {dataset_name} dataset...")
#     print(f"Total documents: {len(text_dict)}")
    
#     # Stage 1
#     print("\nStage 1: Classification and relevance...")
#     stage1_results = stage1_classify(text_dict, dataset_name)
    
#     if not stage1_results:
#         print(f"No results from Stage 1 for {dataset_name}")
#         return pd.DataFrame()
    
#     # Stage 2
#     print("\nStage 2: Detailed scoring...")
#     stage2_results = stage2_score(stage1_results, text_dict, dataset_name)
    
#     # Combine
#     print("\nCombining results...")
#     final_results = combine_results(stage1_results, stage2_results, dataset_name)
    
#     if len(final_results) > 0:
#         # Print summary statistics
#         print(f"\nSummary for {dataset_name}:")
#         print(f"Total documents: {len(final_results)}")
#         print(f"Relevant documents: {sum(1 for _, row in final_results.iterrows() if row.get('is_relevant') == 'yes')}")
        
#         # Only calculate means if there are numeric values
#         if 'cost_reduction_score' in final_results.columns:
#             print(f"Average cost score: {final_results['cost_reduction_score'].mean():.2f}")
#         if 'efficiency_gain_score' in final_results.columns:
#             print(f"Average efficiency score: {final_results['efficiency_gain_score'].mean():.2f}")
#         if 'circularity_score' in final_results.columns:
#             print(f"Average circularity score: {final_results['circularity_score'].mean():.2f}")
    
#     return final_results

# # Optional: Add ISO heuristic score for comparison
# def add_iso_heuristic_score(results_df, text_dict):
#     """Add ISO heuristic score for comparison with LLM scoring"""
#     if len(results_df) > 0:
#         results_df['iso_heuristic_score'] = results_df.index.map(
#             lambda x: iso_score(text_dict.get(x, ''))
#         )
#     return results_df

# # Test function with proper error handling
# def test_small_batch(text_dict, n=5):
#     """Test with a small number of documents to verify the pipeline works"""
#     print(f"\n=== Testing with {n} documents ===")
    
#     # Get first n documents
#     test_docs = dict(list(text_dict.items())[:n])
    
#     print("Test documents:")
#     for doc_id, text in test_docs.items():
#         print(f"- {doc_id}: {text[:100]}...")
    
#     # Process
#     results = stage1_classify(test_docs, "test_small")
    
#     if results:
#         print("\nResults:")
#         for doc_id, result in results.items():
#             print(f"- {doc_id}: relevant={result.get('is_relevant')}, category={result.get('category')}")
    
#     return results

# # Check outputs directory function
# def check_output_files():
#     """Check what files exist in the outputs directory"""
#     output_files = list(OUTPUT_DIR.glob("*.jsonl"))
#     print(f"\nFiles in outputs directory:")
#     for f in output_files:
#         size = f.stat().st_size
#         lines = count_lines_in_file(f) if size > 0 else 0
#         print(f"- {f.name}: {size:,} bytes, {lines} lines")
#     return output_files

# # === MAIN EXECUTION ===

# # First, apply keyword filtering to both datasets
# print("\n=== Applying Keyword Filtering ===")
# patent_subset = {k: v for k, v in patent_dict.items() if contains_keyword(v)}
# print(f"Patents after keyword filtering: {len(patent_subset):,} out of {len(patent_dict):,}")

# paper_subset = {k: v for k, v in paper_dict.items() if contains_keyword(v)}
# print(f"Papers after keyword filtering: {len(paper_subset):,} out of {len(paper_dict):,}")

# # Test with a small batch first
# print("\n=== Running Small Test First ===")
# test_results = test_small_batch(patent_subset, n=3)

# if test_results:
#     print("\n=== Test successful! ===")
    
#     # Ask user if they want to proceed with full processing
#     user_input = input("\nProceed with full dataset processing? (y/n): ")
    
#     if user_input.lower() == 'y':
#         # Process patents
#         if len(patent_subset) > 1000:
#             print("\nProcessing large patent dataset in batches...")
#             patent_batch1 = dict(list(patent_subset.items())[:1000])
#             patent_results = process_dataset_two_stage(patent_batch1, "patents_v5_batch1")
#         else:
#             patent_results = process_dataset_two_stage(patent_subset, "patents_v5")
        
#         # Process papers
#         if len(paper_subset) > 500:
#             print("\nProcessing large paper dataset in batches...")
#             paper_batch1 = dict(list(paper_subset.items())[:500])
#             paper_results = process_dataset_two_stage(paper_batch1, "papers_v5_batch1")
#         else:
#             paper_results = process_dataset_two_stage(paper_subset, "papers_v5")
        
#         # Add ISO heuristic scores
#         if len(patent_results) > 0:
#             patent_results = add_iso_heuristic_score(patent_results, patent_dict)
#             print("\nSample patent results:")
#             print(patent_results.head())
        
#         if len(paper_results) > 0:
#             paper_results = add_iso_heuristic_score(paper_results, paper_dict)
#             print("\nSample paper results:")
#             print(paper_results.head())
        
#         print("\n=== Processing Complete! ===")
#     else:
#         print("Processing cancelled by user.")
# else:
#     print("\n=== Test failed! Check your API configuration ===")
#     print("Make sure your OpenAI API key is set in your environment variables.")

# # Always check output files at the end
# check_output_files()

In [27]:
# ## 7 – Two-Stage Processing with Direct OpenAI API

# import time
# import json
# from openai import OpenAI

# OUTPUT_DIR = PROJECT_DIR / "outputs"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# # Initialize OpenAI client
# client = OpenAI()

# # Stage 1: Classification and Relevance
# def stage1_classify_direct(data_dict, dataset_name):
#     """Stage 1: Basic classification and relevance using direct API"""
    
#     print(f"\nStage 1: Classifying {len(data_dict)} documents...")
    
#     CLASSIFY_SYSTEM = f"""
# You are classifying heat pump technology documents.

# CATEGORIES:
# {json.dumps(CATEGORIES, indent=2)}

# CATEGORY EXPLANATIONS:
# {json.dumps(CATEGORY_EXPLANATIONS, indent=2)}

# INSTRUCTIONS:
# 1. Determine if the document is PRIMARILY about heat pump innovation or technology
# 2. If relevant, assign the most appropriate category
# 3. Provide a brief summary of the innovation
# 4. List key technical innovations mentioned

# Return JSON with these fields only.
# """
    
#     classify_fields = [
#         {"name": "is_relevant", "type": "str", 
#          "description": "'yes' if primarily about heat pump innovation, 'no' otherwise"},
#         {"name": "relevance_reason", "type": "str", 
#          "description": "Brief explanation (max 20 words) of relevance decision"},
#         {"name": "category", "type": "str", 
#          "description": f"Primary category from: {', '.join(CATEGORIES.values())}. Use 'N/A' if not relevant."},
#         {"name": "summary", "type": "str", 
#          "description": "Brief summary (≤25 words) of the innovation. Use 'N/A' if not relevant."},
#         {"name": "key_innovations", "type": "list[str]", 
#          "description": "List of specific technical features mentioned"},
#     ]
    
#     results = []
#     output_path = OUTPUT_DIR / f"{dataset_name}_stage1_classified.jsonl"
    
#     # Process documents
#     for i, (doc_id, text) in enumerate(data_dict.items()):
#         if i % 10 == 0:
#             print(f"  Processing document {i+1}/{len(data_dict)}...")
        
#         try:
#             # Create prompt
#             field_desc = "\n".join([f"- {f['name']}: {f['description']}" for f in classify_fields])
            
#             response = client.chat.completions.create(
#                 model="gpt-4o-mini",
#                 messages=[
#                     {"role": "system", "content": CLASSIFY_SYSTEM},
#                     {"role": "user", "content": f"""
# Analyze this document:

# {text[:1500]}

# Provide JSON response with:
# {field_desc}
# """}
#                 ],
#                 response_format={"type": "json_object"},
#                 temperature=0
#             )
            
#             result = json.loads(response.choices[0].message.content)
#             result['_id'] = doc_id
#             results.append(result)
            
#             # Save incrementally
#             if i % 10 == 0:
#                 temp_df = pd.DataFrame(results)
#                 temp_df.to_json(output_path, orient='records', lines=True)
            
#         except Exception as e:
#             print(f"  Error processing {doc_id}: {e}")
#             # Add minimal result for failed documents
#             results.append({
#                 '_id': doc_id,
#                 'is_relevant': 'error',
#                 'relevance_reason': str(e)[:50],
#                 'category': 'N/A',
#                 'summary': 'Error processing document',
#                 'key_innovations': []
#             })
        
#         # Rate limiting
#         time.sleep(0.5)
    
#     # Save final results
#     results_df = pd.DataFrame(results)
#     results_df.to_json(output_path, orient='records', lines=True)
#     print(f"Stage 1 complete: {len(results_df)} documents processed")
#     print(f"Saved to: {output_path}")
    
#     return results_df.set_index('_id').to_dict('index')

# # Stage 2: Detailed Scoring
# def stage2_score_direct(classified_results, original_texts, dataset_name):
#     """Stage 2: Detailed scoring of relevant documents using direct API"""
    
#     # Filter only relevant documents
#     relevant_ids = {doc_id: results for doc_id, results in classified_results.items() 
#                    if results.get('is_relevant') == 'yes'}
    
#     if not relevant_ids:
#         print("No relevant documents found in Stage 1")
#         return {}
    
#     print(f"\nStage 2: Scoring {len(relevant_ids)} relevant documents...")
    
#     SCORE_SYSTEM = f"""
# You are scoring heat pump innovations for their potential impact.

# Technology Readiness Levels (TRL):
# {TRL_DEFINITIONS}

# Application Type Indicators:
# {format_application_indicators(APPLICATION_INDICATORS)}

# ISO 59004 Circularity Principles:
# - Maintain: predictive maintenance, condition monitoring, service models
# - Reuse: remanufacturing, refurbishment, modular replacement
# - Recycle: material recovery, closed-loop systems
# - Reduce: efficient manufacturing, minimal waste
# - Regenerate: bio-based materials, renewable resources

# SCORING CRITERIA:
# Cost Reduction (0-3):
# - 0: No cost reduction potential
# - 1: Minor reduction (<10% cost savings)
# - 2: Moderate reduction (10-30% cost savings)
# - 3: Major reduction (>30% cost savings)

# Efficiency Gain (0-3):
# - 0: No efficiency improvement
# - 1: Minor gain (<10% COP improvement)
# - 2: Moderate gain (10-30% COP improvement)
# - 3: Major gain (>30% COP improvement)

# INSTRUCTIONS:
# Based on the document, assess the innovation's potential and maturity.
# Consider both explicit claims and reasonable inferences from the technology described.
# """
    
#     score_fields = [
#         {"name": "application_type", "type": "str", 
#          "description": "'domestic', 'industrial', 'both', or 'unclear'"},
#         {"name": "specific_applications", "type": "list[str]", 
#          "description": "Specific use cases (e.g., 'space heating', 'water heating')"},
#         {"name": "trl_level", "type": "int", 
#          "description": "TRL 1-9 based on development stage described"},
#         {"name": "trl_evidence", "type": "str", 
#          "description": "Evidence supporting TRL assessment"},
#         {"name": "has_cost_reduction_potential", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "cost_reduction_reason", "type": "str", 
#          "description": "How it reduces costs (if applicable)"},
#         {"name": "cost_reduction_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
#         {"name": "has_efficiency_gain", "type": "str", 
#          "description": "'yes' or 'no'"},
#         {"name": "efficiency_gain_reason", "type": "str", 
#          "description": "How it improves efficiency (if applicable)"},
#         {"name": "efficiency_gain_score", "type": "int", 
#          "description": "Score 0-3 based on potential magnitude"},
#         {"name": "circularity_principles", "type": "list[str]", 
#          "description": "Which principles apply: 'maintain', 'reuse', 'recycle', 'reduce', 'regenerate'"},
#         {"name": "circularity_evidence", "type": "str", 
#          "description": "Specific circular features mentioned"},
#         {"name": "circularity_score", "type": "int", 
#          "description": "0-3 based on number of principles"},
#         {"name": "materials_mentioned", "type": "list[str]", 
#          "description": "Specific materials or components mentioned"},
#     ]
    
#     results = []
#     output_path = OUTPUT_DIR / f"{dataset_name}_stage2_scored.jsonl"
    
#     # Process relevant documents
#     for i, doc_id in enumerate(relevant_ids.keys()):
#         if i % 10 == 0:
#             print(f"  Scoring document {i+1}/{len(relevant_ids)}...")
        
#         try:
#             # Get original text and Stage 1 summary
#             text = original_texts[doc_id]
#             stage1_info = classified_results[doc_id]
            
#             # Create prompt
#             field_desc = "\n".join([f"- {f['name']}: {f['description']}" for f in score_fields])
            
#             response = client.chat.completions.create(
#                 model="gpt-4o-mini",
#                 messages=[
#                     {"role": "system", "content": SCORE_SYSTEM},
#                     {"role": "user", "content": f"""
# Analyze this heat pump innovation:

# Category: {stage1_info.get('category', 'Unknown')}
# Summary: {stage1_info.get('summary', 'No summary')}

# Full document (first 2000 chars):
# {text[:2000]}

# Provide JSON response with:
# {field_desc}
# """}
#                 ],
#                 response_format={"type": "json_object"},
#                 temperature=0
#             )
            
#             result = json.loads(response.choices[0].message.content)
#             result['_id'] = doc_id
#             results.append(result)
            
#             # Save incrementally
#             if i % 10 == 0:
#                 temp_df = pd.DataFrame(results)
#                 temp_df.to_json(output_path, orient='records', lines=True)
            
#         except Exception as e:
#             print(f"  Error scoring {doc_id}: {e}")
#             # Add default scores for failed documents
#             results.append({
#                 '_id': doc_id,
#                 'application_type': 'unclear',
#                 'specific_applications': [],
#                 'trl_level': 0,
#                 'trl_evidence': f'Error: {str(e)[:50]}',
#                 'has_cost_reduction_potential': 'no',
#                 'cost_reduction_reason': 'Error processing',
#                 'cost_reduction_score': 0,
#                 'has_efficiency_gain': 'no',
#                 'efficiency_gain_reason': 'Error processing',
#                 'efficiency_gain_score': 0,
#                 'circularity_principles': [],
#                 'circularity_evidence': 'Error processing',
#                 'circularity_score': 0,
#                 'materials_mentioned': []
#             })
        
#         # Rate limiting
#         time.sleep(0.5)
    
#     # Save final results
#     results_df = pd.DataFrame(results)
#     results_df.to_json(output_path, orient='records', lines=True)
#     print(f"Stage 2 complete: {len(results_df)} documents scored")
#     print(f"Saved to: {output_path}")
    
#     return results_df.set_index('_id').to_dict('index')

# # Combine results function
# def combine_results(stage1_results, stage2_results, dataset_name):
#     """Combine results from both stages into final output"""
    
#     combined = {}
    
#     for doc_id, stage1 in stage1_results.items():
#         if stage1.get('is_relevant') == 'yes' and doc_id in stage2_results:
#             # Combine relevant documents with both stages
#             combined[doc_id] = {**stage1, **stage2_results[doc_id]}
#         else:
#             # Keep non-relevant documents with stage 1 info only
#             combined[doc_id] = stage1
#             # Add empty fields for consistency
#             combined[doc_id].update({
#                 'application_type': 'N/A',
#                 'specific_applications': [],
#                 'trl_level': 0,
#                 'trl_evidence': 'N/A',
#                 'has_cost_reduction_potential': 'no',
#                 'cost_reduction_reason': 'N/A',
#                 'cost_reduction_score': 0,
#                 'has_efficiency_gain': 'no',
#                 'efficiency_gain_reason': 'N/A',
#                 'efficiency_gain_score': 0,
#                 'circularity_principles': [],
#                 'circularity_evidence': 'N/A',
#                 'circularity_score': 0,
#                 'materials_mentioned': []
#             })
    
#     # Save combined results
#     combined_df = pd.DataFrame.from_dict(combined, orient='index')
#     combined_df.index.name = '_id'
    
#     # Save to file
#     output_path = OUTPUT_DIR / f"{dataset_name}_final_combined.jsonl"
#     combined_df.reset_index().to_json(
#         output_path,
#         orient='records',
#         lines=True
#     )
    
#     print(f"\nSaved combined results to: {output_path}")
    
#     return combined_df

# # Main processing function
# def process_dataset_two_stage(text_dict, dataset_name, max_docs=None):
#     """Process a dataset through both stages using direct API"""
    
#     # Limit documents if specified
#     if max_docs:
#         text_dict = dict(list(text_dict.items())[:max_docs])
    
#     print(f"\n{'='*50}")
#     print(f"Processing {dataset_name} dataset")
#     print(f"Total documents: {len(text_dict)}")
#     print(f"{'='*50}")
    
#     # Stage 1
#     stage1_results = stage1_classify_direct(text_dict, dataset_name)
    
#     if not stage1_results:
#         print(f"No results from Stage 1 for {dataset_name}")
#         return pd.DataFrame()
    
#     # Stage 2
#     stage2_results = stage2_score_direct(stage1_results, text_dict, dataset_name)
    
#     # Combine
#     print("\nCombining results...")
#     final_results = combine_results(stage1_results, stage2_results, dataset_name)
    
#     # Add ISO heuristic score
#     print("Adding ISO heuristic scores...")
#     final_results['iso_heuristic_score'] = final_results.index.map(
#         lambda x: iso_score(text_dict.get(x, ''))
#     )
    
#     # Print summary
#     if len(final_results) > 0:
#         print(f"\n{'='*50}")
#         print(f"Summary for {dataset_name}:")
#         print(f"{'='*50}")
#         print(f"Total documents: {len(final_results)}")
        
#         relevant_count = sum(1 for _, row in final_results.iterrows() if row.get('is_relevant') == 'yes')
#         print(f"Relevant documents: {relevant_count} ({relevant_count/len(final_results)*100:.1f}%)")
        
#         if relevant_count > 0:
#             relevant_df = final_results[final_results['is_relevant'] == 'yes']
#             print(f"\nCategory distribution (relevant docs):")
#             cat_dist = relevant_df['category'].value_counts().head(10)
#             for cat, count in cat_dist.items():
#                 print(f"  {cat}: {count}")
            
#             print(f"\nScore averages (relevant docs):")
#             print(f"  Cost reduction: {relevant_df['cost_reduction_score'].mean():.2f}")
#             print(f"  Efficiency gain: {relevant_df['efficiency_gain_score'].mean():.2f}")
#             print(f"  Circularity: {relevant_df['circularity_score'].mean():.2f}")
#             print(f"  ISO heuristic: {relevant_df['iso_heuristic_score'].mean():.2f}")
            
#             print(f"\nTRL distribution (relevant docs):")
#             trl_dist = relevant_df['trl_level'].value_counts().sort_index()
#             for trl, count in trl_dist.items():
#                 print(f"  TRL {trl}: {count}")
    
#     return final_results

# # === MAIN EXECUTION ===

# print("\n" + "="*60)
# print("INDEPENDENT TWO-STAGE PROCESSING PIPELINE")
# print("="*60)

# # Apply keyword filtering
# print("\n1. Applying Keyword Filtering...")
# patent_subset = {k: v for k, v in patent_dict.items() if contains_keyword(v)}
# paper_subset = {k: v for k, v in paper_dict.items() if contains_keyword(v)}

# print(f"   Patents with keywords: {len(patent_subset):,} out of {len(patent_dict):,}")
# print(f"   Papers with keywords: {len(paper_subset):,} out of {len(paper_dict):,}")

# # Test with small batch first
# print("\n2. Testing with small batch...")
# test_batch = dict(list(patent_subset.items())[:3])
# test_results = process_dataset_two_stage(test_batch, "test_v7", max_docs=3)

# if len(test_results) > 0:
#     print("\n✓ Test successful! Sample results:")
#     print(test_results[['is_relevant', 'category', 'trl_level', 'cost_reduction_score']].head())
    
#     # Ask user how to proceed
#     print("\n3. Choose processing option:")
#     print("   a) Process sample (100 patents, 50 papers)")
#     print("   b) Process medium batch (500 patents, 250 papers)")
#     print("   c) Process full datasets (may take hours)")
#     print("   d) Exit")
    
#     choice = input("\nEnter your choice (a/b/c/d): ").lower()
    
#     if choice == 'a':
#         # Small sample
#         patent_results = process_dataset_two_stage(patent_subset, "patents_v7_sample", max_docs=100)
#         paper_results = process_dataset_two_stage(paper_subset, "papers_v7_sample", max_docs=50)
        
#     elif choice == 'b':
#         # Medium batch
#         patent_results = process_dataset_two_stage(patent_subset, "patents_v7_medium", max_docs=500)
#         paper_results = process_dataset_two_stage(paper_subset, "papers_v7_medium", max_docs=250)
        
#     elif choice == 'c':
#         # Full processing
#         print("\nWarning: This will process thousands of documents and may take several hours.")
#         confirm = input("Are you sure? (yes/no): ")
#         if confirm.lower() == 'yes':
#             patent_results = process_dataset_two_stage(patent_subset, "patents_v7_full")
#             paper_results = process_dataset_two_stage(paper_subset, "papers_v7_full")
#         else:
#             print("Full processing cancelled.")
            
#     else:
#         print("Processing cancelled.")
        
# else:
#     print("\n✗ Test failed! Check error messages above.")

# # Check final output files
# print("\n4. Final Output Files:")
# output_files = sorted(OUTPUT_DIR.glob("*_v7_*.jsonl"))
# for f in output_files:
#     lines = count_lines_in_file(f)
#     print(f"   {f.name}: {lines} documents")

# print("\n" + "="*60)
# print("PROCESSING COMPLETE")
# print("="*60)

In [28]:
# path = '/home/pascualdiego/projects/DiscoveryHP/outputs/patents2_classified.jsonl'
# df = pd.read_json(path, lines=True)

In [29]:
# # 7 – Two‑stage GPT processing pipeline
# """step7_pipeline.py  (v2)

# **Patch:** handles cases where `batch_check` finishes but the JSONL file has
# not appeared (or is still empty) yet, which previously triggered
# `ValueError: Expected object or value` in `pandas.read_json()`.

# Changes
# -------
# * Added `_wait_for_jsonl()` helper that polls for file creation & non‑zero size.
# * `run_llm_pipeline()` now waits (default 180 s) after each stage and raises a
#   clear `RuntimeError` if the file is still missing/empty.
# * Reads are wrapped in `try/except` to provide friendlier diagnostics.
# * No other logic touched, so previous API (`run_default_datasets`, etc.) is
#   unchanged.
# """
# from __future__ import annotations

# import json
# import math
# import time
# from inspect import signature
# from pathlib import Path
# from typing import Dict, List

# import pandas as pd

# try:
#     from tqdm.auto import tqdm  # optional; only imported if available
# except ModuleNotFoundError:  # pragma: no cover – tqdm optional
#     tqdm = None  # type: ignore

# # ---------------------------------------------------------------------------
# # 0 – Imports that are already available in the notebook environment
# # ---------------------------------------------------------------------------
# try:
#     from discovery_heat_pump_futures import PROJECT_DIR  # project helper
# except ModuleNotFoundError:
#     PROJECT_DIR = Path(".").resolve()

# from discovery_utils.utils.llm import batch_check  # batched GPT helper

# # ---------------------------------------------------------------------------
# # 1 – ISO‑59004 heuristic (unchanged)
# # ---------------------------------------------------------------------------
# import re
# from nltk.stem import WordNetLemmatizer

# lemmatizer = WordNetLemmatizer()
# _token_re = re.compile(r"\b\w+\b")

# ISO_LEVERS = {
#     "maintain": [
#         "predictive maintenance", "preventive maintenance", "condition monitoring",
#         "remote diagnostics", "fault detection", "service-as-a-product", "self-healing system",
#     ],
#     "reuse": [
#         "remanufacture", "remanufacturing", "refurbish", "refurbished", "repair", "repairability",
#         "cartridge", "replaceable module", "core exchange", "component reuse", "second life",
#     ],
#     "recycle": [
#         "recycled", "recycling", "reclaim", "reclaimed", "material recovery",
#         "mono-material", "material loop", "mechanical recycling", "chemical recycling", "closed-loop",
#     ],
#     "reduce": [
#         "near-net shape", "additive manufacturing", "3d printing", "lightweight design",
#         "material efficiency", "yield improvement", "low-waste", "minimal material use", "net-shape forming",
#     ],
#     "regenerate": [
#         "bio-based", "biobased", "bio-polymer", "biopolymer", "biocomposite",
#         "renewable feedstock", "natural material", "biodegradable", "biomaterial", "plant-derived",
#     ],
# }

# def _norm(text: str) -> List[str]:
#     return [lemmatizer.lemmatize(w) for w in _token_re.findall(text.lower())]

# def iso_score(text: str) -> int:
#     toks = " ".join(_norm(text))
#     hits = []
#     for kws in ISO_LEVERS.values():
#         m = sum(1 for kw in kws if kw in toks)
#         hits.append(0 if m == 0 else 1 if m == 1 else 2 if m == 2 else 3)
#     return math.ceil(sum(hits) / 5)

# # ---------------------------------------------------------------------------
# # 2 – Prompt templates and field specs (truncated for brevity ‑‑ unchanged)
# # ---------------------------------------------------------------------------
# try:
#     CATEGORIES
# except NameError:
#     CATEGORIES = {}
# try:
#     CATEGORY_EXPLANATIONS
# except NameError:
#     CATEGORY_EXPLANATIONS = {}
# try:
#     TRL_DEFINITIONS
# except NameError:
#     TRL_DEFINITIONS = "TRL definitions missing — run earlier cells."
# try:
#     APPLICATION_INDICATORS
# except NameError:
#     APPLICATION_INDICATORS = {}

# def _fmt_app_indicators(dic: Dict[str, List[str]]) -> str:
#     return "\n".join([f"- {k.capitalize()}: {', '.join(v)}" for k, v in dic.items()])

# STAGE1_SYSTEM = """You are classifying heat‑pump technology documents. …"""
# STAGE1_FIELDS = [
#     {"name": "is_relevant", "type": "str", "description": "'yes' or 'no'"},
#     {"name": "relevance_reason", "type": "str", "description": "≤20 words"},
#     {"name": "category", "type": "str", "description": "Primary category or 'N/A'"},
#     {"name": "summary", "type": "str", "description": "≤25 words"},
#     {"name": "key_innovations", "type": "list[str]", "description": "Innovation list"},
# ]

# STAGE2_SYSTEM = """You are scoring heat‑pump innovations. …"""
# STAGE2_FIELDS = [
#     {"name": "application_type", "type": "str", "description": "domestic/industrial/both/unclear"},
#     # … (rest identical to previous version)
# ]

# # ---------------------------------------------------------------------------
# # 3 – Helper: safe run() wrapper + file‑waiter
# # ---------------------------------------------------------------------------

# def _processor_run(proc, data, *, batch_size: int, sleep_time: float, progress: bool):
#     """Call `proc.run()` with only supported kwargs (compat fix)."""
#     kwargs = {"batch_size": batch_size, "sleep_time": sleep_time}
#     if "show_progress" in signature(proc.run).parameters:
#         kwargs["show_progress"] = progress
#     proc.run(data, **kwargs)

# def _wait_for_jsonl(path: Path, timeout: int = 180, min_bytes: int = 20):
#     """Poll until *path* exists and is > *min_bytes* (otherwise raise)."""
#     start = time.time()
#     while time.time() - start < timeout:
#         if path.exists() and path.stat().st_size >= min_bytes:
#             return
#         time.sleep(2)
#     raise RuntimeError(f"Timed‑out waiting for {path.name} to be written by LLMProcessor.")

# # ---------------------------------------------------------------------------
# # 4 – Core pipeline
# # ---------------------------------------------------------------------------

# def _ensure_id_column(df: pd.DataFrame) -> pd.DataFrame:
#     if "_id" not in df.columns:
#         df["_id"] = df.get("id", df.index)
#     return df.set_index("_id")

# def run_llm_pipeline(
#     text_dict: Dict[str, str],
#     dataset_name: str,
#     *,
#     batch_size: int = 30,
#     model: str = "gpt-4o-mini",
#     progress: bool = True,
# ):
#     output_dir: Path = PROJECT_DIR / "outputs"
#     output_dir.mkdir(parents=True, exist_ok=True)

#     # ---------- Stage 1 ----------
#     s1_path = output_dir / f"{dataset_name}_stage1.jsonl"
#     proc1 = batch_check.LLMProcessor(
#         model_name=model,
#         temperature=0,
#         system_message=STAGE1_SYSTEM,
#         session_name=f"{dataset_name}_s1",
#         output_fields=STAGE1_FIELDS,
#         output_path=str(s1_path),
#     )
#     _processor_run(proc1, text_dict, batch_size=batch_size, sleep_time=0.4, progress=progress)
#     _wait_for_jsonl(s1_path)
#     try:
#         s1_df = _ensure_id_column(pd.read_json(s1_path, lines=True))
#     except ValueError as e:
#         raise RuntimeError(f"Failed to parse Stage‑1 output {s1_path}: {e}") from e

#     relevant_ids = s1_df[s1_df["is_relevant"] == "yes"].index.tolist()
#     if not relevant_ids:
#         s1_df["iso_heuristic_score"] = s1_df.index.map(lambda i: iso_score(text_dict[i]))
#         return s1_df

#     # ---------- Stage 2 ----------
#     s2_path = output_dir / f"{dataset_name}_stage2.jsonl"
#     proc2 = batch_check.LLMProcessor(
#         model_name=model,
#         temperature=0,
#         system_message=STAGE2_SYSTEM,
#         session_name=f"{dataset_name}_s2",
#         output_fields=STAGE2_FIELDS,
#         output_path=str(s2_path),
#     )
#     _processor_run(proc2, {i: text_dict[i] for i in relevant_ids},
#                    batch_size=batch_size, sleep_time=0.4, progress=progress)
#     _wait_for_jsonl(s2_path)
#     try:
#         s2_df = _ensure_id_column(pd.read_json(s2_path, lines=True))
#     except ValueError as e:
#         raise RuntimeError(f"Failed to parse Stage‑2 output {s2_path}: {e}") from e

#     # ---------- Merge + ISO ----------
#     combined = s1_df.join(s2_df, how="left")
#     combined["iso_heuristic_score"] = combined.index.map(lambda i: iso_score(text_dict[i]))

#     final_path = output_dir / f"{dataset_name}_final.jsonl"
#     combined.reset_index().to_json(final_path, orient="records", lines=True)
#     print(f"[Step 7] Saved {len(combined)} records → {final_path.relative_to(PROJECT_DIR)}")
#     return combined

# # ---------------------------------------------------------------------------
# # 5 – Convenience: run default datasets (unchanged API)
# # ---------------------------------------------------------------------------

# def run_default_datasets(*, keyword_filter: bool = True, batch_size: int = 30, progress: bool = True):
#     globals_ = globals()
#     if "patent_dict" not in globals_ or "paper_dict" not in globals_:
#         raise RuntimeError("patent_dict / paper_dict not found – run earlier cells.")

#     patent_dict = globals_["patent_dict"]
#     paper_dict = globals_["paper_dict"]

#     if keyword_filter and "contains_keyword" in globals_:
#         contains_keyword = globals_["contains_keyword"]
#         patent_subset = {k: v for k, v in patent_dict.items() if contains_keyword(v)}
#         paper_subset = {k: v for k, v in paper_dict.items() if contains_keyword(v)}
#     else:
#         patent_subset = patent_dict
#         paper_subset = paper_dict

#     print(f"[Step 7] Processing {len(patent_subset):,} patent docs · {len(paper_subset):,} paper docs")

#     patent_df = run_llm_pipeline(patent_subset, "patents_v8", batch_size=batch_size, progress=progress)
#     paper_df  = run_llm_pipeline(paper_subset,  "papers_v8",  batch_size=batch_size, progress=progress)
#     return patent_df, paper_df

# # ---------------------------------------------------------------------------
# # 6 – Script execution guard
# # ---------------------------------------------------------------------------
# if __name__ == "__main__":
#     run_default_datasets()




[Step 7] Processing 15,847 patent docs · 12,511 paper docs
2025-05-27 10:22:26,707 - root - INFO - Using OpenAI
2025-05-27 10:22:26,783 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client
2025-05-27 10:22:27,053 - root - INFO - Using OpenAI
2025-05-27 10:22:27,102 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client


KeyboardInterrupt: 

2025-05-27 10:24:36,719 - root - INFO - All data has already been processed.
2025-05-27 10:24:36,728 - root - INFO - Processing batch 1/511
2025-05-27 10:24:40,215 - root - INFO - Processing batch 2/511
2025-05-27 10:24:41,899 - root - INFO - Processing batch 3/511
2025-05-27 10:24:44,142 - root - INFO - Processing batch 4/511
2025-05-27 10:24:46,463 - root - INFO - Processing batch 5/511
2025-05-27 10:24:48,575 - root - INFO - Processing batch 6/511
2025-05-27 10:24:50,752 - root - INFO - Processing batch 7/511
2025-05-27 10:24:52,301 - root - INFO - Processing batch 8/511
2025-05-27 10:24:54,246 - root - INFO - Processing batch 9/511
2025-05-27 10:24:58,630 - root - INFO - Processing batch 10/511
2025-05-27 10:25:01,347 - root - INFO - Processing batch 11/511
2025-05-27 10:25:03,768 - root - INFO - Processing batch 12/511
2025-05-27 10:25:15,330 - root - INFO - Processing batch 13/511
2025-05-27 10:25:20,588 - root - INFO - Processing batch 14/511
2025-05-27 10:25:23,268 - root - INF